# Ablation 05 — Concept Faithfulness vs clinical labels (MeSH/Problems)

**Internal tag: `a5`.** Analysis-only, no retraining. Reuses the baseline SAE
checkpoint (seed 42, dict4096, k=32) and evaluates whether the discovered
features are faithful to real per-image clinical labels.

Ablations 00–04 showed that concepts are unstable cross-seed
(Jaccard ~ chance floor, consensus ~0). But instability ≠ uselessness: the
concepts that *exist* (in a given seed) are meaningful, i.e. do they activate
on images that actually contain a certain pathology/anatomy?

ab05 encodes the test images → activation matrix `A` (1494 × 4096), builds a
binary label matrix `Y` from the `MeSH`/`Problems` columns of
`indiana_reports.csv`, and computes the point-biserial correlation between each
feature and each label. A feature is "faithful" if it beats a calibrated null
(shuffle + FDR).

**Result (preview).** ~10% of *live* features beat their own per-feature null
(p95). The most faithful ones track visually concrete concepts
(implants/defibrillators, pleural effusion, emphysema, cardiac shadow, edema).
Concepts are unstable cross-seed (a0–a4) but, when they exist, they are
moderately faithful to real labels (a5).

In [1]:
# Reproducibility env vars — MUST be set before importing torch.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("PYTHONHASHSEED", "0")

import sys, csv, json, re, math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

# Resolve project root (walk up until 'src/' exists)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f'Project root: {PROJECT_ROOT}')
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f'Device: {device}')


Project root: /Users/marcantoniolopez/Documents/github/xai-project-5
Device: mps


In [2]:
import config
import utils
from scipy.stats import norm

# OUTPUT-DIR ISOLATION (hard rule #2) — rebind the mutable PathsConfig in place.
ABLATION_RESULTS_DIR = PROJECT_ROOT / 'results' / 'ablation'
ABLATION_FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures' / 'ablation'
config.paths.results_dir = ABLATION_RESULTS_DIR
config.paths.figures_dir = ABLATION_FIGURES_DIR
ABLATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Data locations for the clinical labels (already on disk).
REPORTS_CSV  = PROJECT_ROOT / 'data' / 'iu_xray' / 'reports' / 'indiana_reports.csv'
PROJ_CSV     = PROJECT_ROOT / 'data' / 'iu_xray' / 'reports' / 'indiana_projections.csv'

print('=== Ablation 05 (faithfulness) — isolated output dirs ===')
print(f'Results dir : {ABLATION_RESULTS_DIR}')
print(f'Figures dir : {ABLATION_FIGURES_DIR}')
print(f'SAE defaults: dict_size={config.sae.dict_size}, k={config.sae.k}, '
      f'activation_dim={config.sae.activation_dim}')
print(f'reports CSV : {REPORTS_CSV} (exists={REPORTS_CSV.exists()})')


/Users/marcantoniolopez/Documents/github/xai-project-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Ablation 05 (faithfulness) — isolated output dirs ===
Results dir : /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation
Figures dir : /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation
SAE defaults: dict_size=4096, k=32, activation_dim=512
reports CSV : /Users/marcantoniolopez/Documents/github/xai-project-5/data/iu_xray/reports/indiana_reports.csv (exists=True)


In [3]:
from autoencoder.sae_module import SAEManager

# Safe deserialization (rule #4)
test_emb = utils.load_tensor(config.paths.test_embeddings_path)      # (1494, 512)
test_ids = json.load(open(config.paths.test_image_ids_path))         # 1494 PNG basenames
SAE_DIR  = config.paths.models_dir / 'sae_seed42'                    # baseline reference model

print(f'test_emb : {tuple(test_emb.shape)}')
print(f'test_ids : {len(test_ids)}  (e.g. {test_ids[:2]})')
assert len(test_ids) == test_emb.shape[0], 'image_ids and embeddings out of lockstep'

mgr = SAEManager()
mgr.load(SAE_DIR)
print(f'SAE loaded: {SAE_DIR.name} (dict_size={mgr.config["dict_size"]}, k={mgr.config["k"]})')


test_emb : (1494, 512)
test_ids : 1494  (e.g. ['3222_IM-1522-2001.dcm.png', '1280_IM-0187-3001.dcm.png'])
SAE loaded: sae_seed42 (dict_size=4096, k=32)


## Protocol

1. **Clinical labels.** `indiana_reports.csv` has two structured columns per
   report: `MeSH` (MeSH terms, e.g. `Cardiomegaly/borderline`) and `Problems`
   (e.g. `Cardiomegaly;Pulmonary Artery`). I parse them into *base terms*:
   split on `;`, base MeSH on `/`, lowercase, exclude `normal`. → **118 unique
   terms** in-distribution.
2. **Per-image join.** `image_id` (PNG basename) → `uid` via
   `indiana_projections.csv` (with fallback on the numeric prefix of the
   basename) → `MeSH`/`Problems`. I build `Y` aligned to the 1494 test images.
3. **SAE activations.** `mgr.encode(test_emb)` → `A` (1494 × 4096), continuous
   TopK values (32 non-zero per row).
4. **Point-biserial correlation.** For each feature *i* and label *j*:
   $r_{ij} = \mathrm{corr}(A_{:,i},\, Y_{:,j})$. Vectorized as
   `A_zᵀ · Y_z / N` (one matmul). A feature is "faithful" if
   $\max_j |r_{ij}|$ exceeds the null.
5. **Calibrated null.** (a) analytic SE $\approx 1/\sqrt{N}$; (b)
   **per-feature shuffle-null**: 200 permutations of the rows of `Y`, p95 of
   the max|corr| for each feature — a beyond-chance threshold specific to each
   feature; (c) **BH-FDR 0.05** on the $n_{\text{live}} \times n_{\text{label}}$
   tests.

**Prevalence filter.** Labels present in very few images
(e.g. 1–3) produce a degenerate $|r| = 1$ by construction (and inflate the
null). I keep only labels with **prevalence ≥ 10** images → meaningful
correlation and a sensible null. (Features must still beat their own per-feature
null, which corrects for the prevalence distribution.)

In [4]:
def parse_labels(mesh, problems):
    # MeSH/Problems -> set of base terms (split ';', base on '/', lower, excl 'normal').
    out = set()
    for field in (mesh, problems):
        if not field:
            continue
        for item in field.split(';'):
            base = item.strip().split('/')[0].strip().lower()
            if base and base != 'normal':
                out.add(base)
    return out

# uid -> label set
uid2labels = {}
with open(REPORTS_CSV) as f:
    for row in csv.DictReader(f):
        uid2labels[int(row['uid'])] = parse_labels(row.get('MeSH', ''), row.get('Problems', ''))

# filename -> uid (canonical map, prefix fallback)
file2uid = {}
with open(PROJ_CSV) as f:
    for row in csv.DictReader(f):
        file2uid[row['filename']] = int(row['uid'])

def basename_to_uid(bn):
    if bn in file2uid:
        return file2uid[bn]
    m = re.match(r'(\d+)_', bn)
    return int(m.group(1)) if m else None

# global label vocabulary
all_terms = sorted(set().union(*uid2labels.values()))
term2idx = {t: i for i, t in enumerate(all_terms)}
print(f'unique clinical terms (MeSH ∪ Problems, base, excl normal): {len(all_terms)}')

# aligned binary label matrix Y (rows in test-set order)
N = len(test_ids)
Y = np.zeros((N, len(all_terms)), dtype=np.float32)
missing = 0
for i, bn in enumerate(test_ids):
    uid = basename_to_uid(bn)
    if uid is None or uid not in uid2labels:
        missing += 1
        continue
    for t in uid2labels[uid]:
        Y[i, term2idx[t]] = 1.0
print(f'test images: {N} | no-label join: {missing} | '
      f'labels/row mean={Y.sum(1).mean():.2f}, max={int(Y.sum(1).max())}')

# keep only labels present in the test set
label_freq_all = Y.sum(0)
present = np.where(label_freq_all > 0)[0]
Y = Y[:, present]
labels_present = [all_terms[j] for j in present]
print(f'labels present in test set: {len(labels_present)}/{len(all_terms)}')


unique clinical terms (MeSH ∪ Problems, base, excl normal): 118
test images: 1494 | no-label join: 0 | labels/row mean=1.57, max=12
labels present in test set: 101/118


In [5]:
MIN_PREV = 10  # a label needs >= MIN_PREV images for correlation to be meaningful

def zscore_cols(M):
    mu = M.mean(0, keepdims=True)
    sd = M.std(0, keepdims=True)
    sd[sd == 0] = 1.0
    return (M - mu) / sd

# prevalence filter
label_freq = Y.sum(0)
keep = np.where(label_freq >= MIN_PREV)[0]
Yf = Y[:, keep]
labels_f = [labels_present[j] for j in keep]
print(f'prevalence filter (>={MIN_PREV} images): {len(keep)}/{Y.shape[1]} labels kept')
print(f'  prevalences: min={int(label_freq[keep].min())}, '
      f'median={int(np.median(label_freq[keep]))}, max={int(label_freq[keep].max())}')

# encode test through the SAE -> activation matrix A
A = mgr.encode(test_emb).cpu().numpy().astype(np.float32)   # (1494, 4096)
print(f'activation A: {A.shape} | nnz/row mean={(A!=0).sum(1).mean():.1f} | max={A.max():.3f}')

# point-biserial correlation: A_zᵀ · Y_z / N
A_z = zscore_cols(A)
Yf_z = zscore_cols(Yf)
corr = (A_z.T @ Yf_z) / N                  # (4096, n_labels_f)
abs_corr = np.abs(corr)
per_feat_max = abs_corr.max(1)             # (4096,) best |corr| per feature
best_label = abs_corr.argmax(1)

# live vs dead (activation-based: a feature is dead if it never fires on test)
live = (A != 0).any(axis=0)                # (4096,) bool
n_live = int(live.sum())
print(f'\nlive features: {n_live}/{A.shape[1]} ({100*live.mean():.1f}%) '
      f'(dead = {100*(1-live.mean()):.1f}%, consistent with baseline ~44%)')

print('\n% faithful live features (best |corr| over prevalent labels):')
for thr in (0.10, 0.15, 0.20, 0.25, 0.30):
    nl = int((per_feat_max[live] > thr).sum())
    print(f'  |corr|>{thr:.2f}: {nl}/{n_live} live ({100*nl/n_live:.1f}%)')


prevalence filter (>=10 images): 50/101 labels kept
  prevalences: min=10, median=30, max=191


activation A: (1494, 4096) | nnz/row mean=32.0 | max=0.386

live features: 2251/4096 (55.0%) (dead = 45.0%, consistent with baseline ~44%)

% faithful live features (best |corr| over prevalent labels):
  |corr|>0.10: 1210/2251 live (53.8%)
  |corr|>0.15: 576/2251 live (25.6%)
  |corr|>0.20: 213/2251 live (9.5%)
  |corr|>0.25: 82/2251 live (3.6%)
  |corr|>0.30: 22/2251 live (1.0%)


## Null calibration

Three levels, consistent with the null-calibrated style of the program (a0/a2/a3):

- **Analytic SE** $\approx 1/\sqrt{N}$ — standard error of the correlation
  under $H_0$ (feature and label independent). At $N{=}1494$: $|r|>0.10 \approx 3.9\sigma$.
- **Per-feature shuffle-null** — 200 permutations of the rows of `Y`; for each
  feature I take the p95 of the shuffled max|corr|. A feature beats chance if
  its observed max|corr| **beats its own p95**. This is the strongest test:
  feature-specific and corrected for the prevalence distribution.
- **BH-FDR 0.05** on the $n_{\text{live}} \times n_{\text{label}}$ tests —
  corrects for multiple testing on the full (feature, label) matrix.

In [6]:
SEED = 0
N_PERM = 200

# (a) analytic
se = 1.0 / math.sqrt(N)
def two_sided_p(r): return 2 * norm.sf(np.abs(np.asarray(r, dtype=float)) / se)
print(f'analytic null SE(r) = {se:.4f}  '
      f'(|r|>0.10 ~ {0.10/se:.1f} sigma, uncorrected p~{float(two_sided_p(0.10)):.2e})')

# (b) shuffle-null per-feature (live features)
live_idx = np.where(live)[0]
rng = np.random.default_rng(SEED)
null_max = np.empty((N_PERM, A.shape[1]), dtype=np.float32)
for p in range(N_PERM):
    perm = rng.permutation(N)
    Yp_z = zscore_cols(Yf[perm])
    null_max[p] = np.abs((A_z.T @ Yp_z) / N).max(1)
null_per_feat_p95 = np.percentile(null_max, 95, axis=0)
null_live_p95 = null_per_feat_p95[live_idx]
n_null_pass = int((per_feat_max[live_idx] > null_live_p95).sum())
print(f'shuffle-null per-feature (p95): {n_null_pass}/{len(live_idx)} live features '
      f'beat their null (median null p95 = {np.median(null_live_p95):.4f})')

# (c) BH-FDR on the full (live x label) matrix
pvals = two_sided_p(abs_corr[live_idx])
n_tests = pvals.size
order = np.argsort(pvals.ravel())
ranked = pvals.ravel()[order]
fdr = ranked * n_tests / (np.arange(n_tests) + 1)
fdr = np.minimum.accumulate(fdr[::-1])[::-1]
n_sig = int((fdr <= 0.05).sum())
sig_pval = ranked[(fdr <= 0.05)].max() if n_sig else 1.0
sig_r = se * norm.ppf(1 - sig_pval / 2)
print(f'BH-FDR 0.05 over {n_tests} tests: {n_sig} significant (feature,label) pairs; '
      f'approx |r| threshold {sig_r:.3f}')

NULL = {
    'n_test': N, 'analytic_se': float(se),
    'shuffle_n_perm': N_PERM, 'shuffle_median_null_p95': float(np.median(null_live_p95)),
    'n_live_beat_null_p95': n_null_pass, 'n_live': n_live,
    'fdr_alpha': 0.05, 'fdr_n_tests': int(n_tests), 'fdr_n_sig': n_sig,
    'fdr_approx_r_threshold': float(sig_r),
}


analytic null SE(r) = 0.0259  (|r|>0.10 ~ 3.9 sigma, uncorrected p~1.11e-04)


shuffle-null per-feature (p95): 226/2251 live features beat their null (median null p95 = 0.1881)
BH-FDR 0.05 over 112550 tests: 3496 significant (feature,label) pairs; approx |r| threshold 0.082


In [7]:
# Cross-check: the SAE's OWN (gap-corrected) name for each faithful feature vs its label.
vocab_emb = utils.load_tensor(config.paths.vocab_embeddings_path)   # (508, 512)
with open(config.paths.vocab_labels_path) as f:
    vocab_labels = [e['term'] if isinstance(e, dict) else e for e in json.load(f)]
train_emb = utils.load_tensor(config.paths.train_embeddings_path)
gap = train_emb.mean(0) - vocab_emb.mean(0)
W_dec = mgr.get_decoder_weights().cpu()                # (4096, 512) — naming on CPU
W_dec_gc = W_dec - gap.unsqueeze(0)
Wn = F.normalize(W_dec_gc, dim=1); Wn[~live] = 0.0
Vn = F.normalize(vocab_emb, dim=1)
name_sim = (Wn @ Vn.T).cpu().numpy()                   # (4096, 508)

def sae_name(feat):
    if not live[feat]:
        return '(dead)', 0.0
    j = int(name_sim[feat].argmax())
    return vocab_labels[j], float(name_sim[feat, j])

# top faithful live features (with label prevalence + SAE own name)
print('TOP 15 live features by max|corr|  (label prevalence | SAE gap-corrected name):')
order = np.argsort(per_feat_max)[::-1]
top_rows = []
shown = 0
for f in order:
    if live[f] and shown < 15:
        bl = int(best_label[f]); prev = int(Yf[:, bl].sum())
        nm, ns = sae_name(f)
        print(f'  feat {f:4d}: |r|={per_feat_max[f]:.3f}  label={labels_f[bl]!r} '
              f'(prev={prev}) | SAE-name={nm!r} (cos={ns:.3f})')
        top_rows.append((int(f), float(per_feat_max[f]), labels_f[bl], prev, nm, round(ns,3)))
        shown += 1

# per-label best feature: can the SAE represent each prevalent clinical concept?
print('\nPER-LABEL best feature (top 12 prevalent labels by best achievable |r|):')
lab_prev = Yf.sum(0)
lab_best = abs_corr.max(0); lab_best_feat = abs_corr.argmax(0)
lo = np.argsort(lab_best)[::-1][:12]
for j in lo:
    print(f"  {labels_f[j]!r} (prev={int(lab_prev[j])}): best |r|={lab_best[j]:.3f} (feat {lab_best_feat[j]})")

TOP 15 live features by max|corr|  (label prevalence | SAE gap-corrected name):
  feat 3785: |r|=0.458  label='implanted medical device' (prev=22) | SAE-name='drug delivery catheter' (cos=0.323)
  feat 2983: |r|=0.349  label='implanted medical device' (prev=22) | SAE-name='balloon catheter' (cos=0.415)
  feat 1261: |r|=0.344  label='implanted medical device' (prev=22) | SAE-name='tunneled central venous catheter without port' (cos=0.484)
  feat  224: |r|=0.342  label='pleural effusion' (prev=57) | SAE-name='ligamentum flavum' (cos=0.350)
  feat 2765: |r|=0.338  label='implanted medical device' (prev=22) | SAE-name='central venous catheter with port or pump' (cos=0.400)
  feat 3962: |r|=0.320  label='pleural effusion' (prev=57) | SAE-name='right diaphragmatic peritoneum' (cos=0.396)
  feat 1253: |r|=0.319  label='infiltrate' (prev=19) | SAE-name='dental device' (cos=0.425)
  feat 3260: |r|=0.315  label='diaphragmatic eventration' (prev=10) | SAE-name='rootlet of spinal nerve' (cos=0.412

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

live_max = per_feat_max[live]
null_live = null_per_feat_p95[live]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# (1) histogram of per-feat max|r| (live) + null overlay
ax = axes[0]
ax.hist(live_max, bins=50, color='#4C72B0', alpha=0.8, label='observed (live features)')
ax.axvline(np.median(null_live), color='#C44E52', ls='--', lw=2,
           label=f'shuffle-null p95 (median={np.median(null_live):.3f})')
for thr, c in [(0.10, '#55A868'), (0.20, '#8172B2')]:
    ax.axvline(thr, color=c, ls=':', lw=1.5, alpha=0.8)
ax.set_xlabel('max $|r_{ij}|$ over prevalent labels')
ax.set_ylabel('# live features')
ax.set_title('Faithfulness distribution vs null')
ax.legend(fontsize=8)

# (2) % faithful live vs threshold, vs null-pass count
ax = axes[1]
thrs = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
pct = [100 * (live_max > t).mean() for t in thrs]
ax.plot(thrs, pct, 'o-', color='#4C72B0', label='% live faithful')
ax.axhline(100 * n_null_pass / n_live, color='#C44E52', ls='--',
           label=f'% beating shuffle-null p95 ({100*n_null_pass/n_live:.1f}%)')
ax.set_xlabel('|corr| threshold')
ax.set_ylabel('% live features')
ax.set_title('Faithful fraction vs threshold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (3) per-label best |r|, colored by prevalence
ax = axes[2]
lab_prev_int = lab_prev.astype(int)
lo_all = np.argsort(lab_best)[::-1][:15][::-1]
sc = ax.barh(range(len(lo_all)), lab_best[lo_all],
             color=plt.cm.viridis(lab_prev_int[lo_all] / lab_prev_int.max()))
ax.set_yticks(range(len(lo_all)))
ax.set_yticklabels([labels_f[j] for j in lo_all], fontsize=8)
ax.axvline(np.median(null_live), color='#C44E52', ls='--', lw=1.5, label='null p95 (median)')
ax.set_xlabel('best $|r|$ (any feature)')
ax.set_title('Best achievable concept↔label alignment')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ABLATION_FIGURES_DIR / 'a5_faithfulness_headline.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'figure -> {ABLATION_FIGURES_DIR / "a5_faithfulness_headline.png"}')


figure -> /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation/a5_faithfulness_headline.png


/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_24614/3068631328.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
def thr_table(live_mask, pfm):
    nl = int(live_mask.sum())
    return {f'{t:.2f}': {'n_live': int((pfm[live_mask] > t).sum()),
                         'pct_live': round(100 * (pfm[live_mask] > t).mean(), 2)}
            for t in (0.10, 0.15, 0.20, 0.25, 0.30)}

out = {
    'ablation': '05_faithfulness',
    'params': {
        'sae_model': 'baseline seed-42 (dict4096, k32)',
        'test_set_size': N,
        'n_clinical_terms_total': len(all_terms),
        'n_labels_present_test': len(labels_present),
        'min_prevalence': MIN_PREV,
        'n_labels_after_prevalence_filter': len(labels_f),
        'prevalence_range': [int(label_freq[keep].min()),
                             int(np.median(label_freq[keep])),
                             int(label_freq[keep].max())],
        'n_live_features': n_live, 'n_total_features': int(A.shape[1]),
        'dead_features_pct': round(100 * (1 - live.mean()), 1),
        'n_perm': N_PERM, 'seed': SEED,
        'metric': 'point-biserial (Pearson) correlation, vectorized A_zᵀ·Y_z/N',
    },
    'faithfulness_by_threshold': thr_table(live, per_feat_max),
    'null_calibration': NULL,
    'top_faithful_features': [
        {'feature_id': r[0], 'best_abs_corr': r[1], 'best_label': r[2],
         'label_prevalence': r[3], 'sae_gap_corrected_name': r[4], 'sae_name_cosine': r[5]}
        for r in top_rows
    ],
    'per_label_best': [
        {'label': labels_f[j], 'prevalence': int(lab_prev[j]),
         'best_abs_corr': round(float(lab_best[j]), 4), 'best_feature': int(lab_best_feat[j])}
        for j in np.argsort(lab_best)[::-1]
    ],
    'headline': {
        'pct_live_beating_shuffle_null_p95': round(100 * n_null_pass / n_live, 2),
        'n_live_beating_shuffle_null_p95': n_null_pass,
        'median_shuffle_null_p95': round(float(np.median(null_live_p95)), 4),
        'strongest_corr': round(float(per_feat_max.max()), 4),
        'strongest_label': labels_f[int(best_label[per_feat_max.argmax()])],
    },
    'interpretation': (
        'Concepts are unstable cross-seed (a0-a4) but, when they exist, a meaningful '
        'minority are faithful to real clinical labels beyond a per-feature shuffle null. '
        'Strongest alignment is on visually concrete concepts (devices, effusion, '
        'emphysema, cardiac shadow, edema); fine-grained pathology is weaker, consistent '
        'with the data-starved regime.'
    ),
}

out_path = ABLATION_RESULTS_DIR / 'a5_faithfulness.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'persisted -> {out_path}')
print(json.dumps(out['headline'], indent=2))


persisted -> /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation/a5_faithfulness.json
{
  "pct_live_beating_shuffle_null_p95": 10.04,
  "n_live_beating_shuffle_null_p95": 226,
  "median_shuffle_null_p95": 0.1881,
  "strongest_corr": 0.4584,
  "strongest_label": "implanted medical device"
}


## Reproducibility notes & status

- Reuses the baseline checkpoint `models/sae_seed42/` (dict4096, k=32), zero training.
- Labels from `indiana_reports.csv` (`MeSH`, `Problems`): 118 unique base
  terms, 101 present in the test set, 50 after the ≥10 prevalence filter.
  `image_id → uid` join via `indiana_projections.csv` (fallback on the numeric
  prefix).
- Point-biserial via a single matmul `A_zᵀ·Y_z/N` instead of ~500k AUROC calls.
- Triple null: analytic SE `1/√N`, per-feature shuffle-null (p95, 200 perm),
  BH-FDR 0.05 on the matrix (live × label). The prevalence filter avoids the
  degenerate `|r|=1` cases of extremely rare labels.
- Correlations only on the 1494 test images; `train_emb` is used only for the
  modality gap shift in the naming cross-reference (as in ab01–04).
- Isolated output: `results/ablation/a5_faithfulness.json` +
  `results/figures/ablation/a5_faithfulness_headline.png`. The baseline is not
  touched.

a0–a4: concepts are unstable cross-seed, and 0.004 is the chance floor.
a5: concepts that *exist* (in seed 42) are moderately but genuinely faithful to
real clinical labels (~10% of live features beat a per-feature null; the
strongest ones track clinically expected concepts). Instability ≠ uselessness.

## References
- [00_consensus.ipynb](00_consensus.ipynb) — a0 (instability in direction space)
- [03_baselines.ipynb](03_baselines.ipynb) — a3 (0.004 = chance floor)
- [../../src/autoencoder/sae_module.py](../../src/autoencoder/sae_module.py) — `SAEManager.encode`
- `data/iu_xray/reports/indiana_reports.csv` — MeSH/Problems columns (in-distribution gold standard)
- [../../results/ablation/a5_faithfulness.json](../../results/ablation/a5_faithfulness.json) — full metrics